# ENGE707 Kaggle Competition: Full-Data ResNet18 and TTA

## Final Model Improvement Experiment

Notebook 5 selected ResNet18 using the untouched validation subset. The first Kaggle submission then scored **0.9463 (94.63%)**, compared with **99.65% validation accuracy**.

This experiment attempts to improve generalisation by:

1. Starting from the best validated ResNet18 checkpoint.
2. Reusing all 63,660 labelled images after model selection is complete.
3. Fine-tuning for three additional epochs with small learning rates.
4. Averaging predictions from the original and horizontally flipped test image.

The test images are used only for prediction. They do not update the model or determine its settings.

## 1. Import the Required Libraries

The notebook uses PyTorch for low-learning-rate fine-tuning and inference, pandas for evidence tables, and Matplotlib for report figures.

In [ ]:
from pathlib import Path
from contextlib import nullcontext
from itertools import chain
from zipfile import ZIP_DEFLATED, ZipFile
import json
import random
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

from IPython.display import FileLink, display

## 2. Locate the Competition Data and Saved Model

The Kaggle version searches all attached inputs, so it does not depend on a particular input-folder name. The original data remains read-only. All new files are written to `/kaggle/working`.

In [ ]:
RUNNING_ON_KAGGLE = Path("/kaggle/input").exists()


def find_kaggle_input_file(filename, required=True):
    """Find one named file among all attached Kaggle inputs."""
    matches = sorted(Path("/kaggle/input").rglob(filename))

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(
            f"Kaggle input file not found: {filename}. "
            "Attach the competition data and Notebook 4 output."
        )

    return None


def find_kaggle_image_directory(filename, split_name):
    """Find the train or test directory containing one image."""
    matches = sorted(Path("/kaggle/input").rglob(filename))
    split_name = split_name.lower()
    split_matches = [
        path for path in matches
        if split_name in {part.lower() for part in path.parent.parts}
    ]

    if not split_matches:
        raise FileNotFoundError(
            f"Could not locate the {split_name} image folder."
        )

    return split_matches[0].parent


if RUNNING_ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    TRAIN_LABELS_PATH = find_kaggle_input_file(
        "train_labels.csv"
    )
    SAMPLE_SUBMISSION_PATH = find_kaggle_input_file(
        "sample_submission.csv"
    )
    PREPROCESSING_CONFIG_PATH = find_kaggle_input_file(
        "preprocessing_config.json"
    )
    BASELINE_CHECKPOINT_PATH = find_kaggle_input_file(
        "resnet18_best.pth"
    )

    train_probe = pd.read_csv(
        TRAIN_LABELS_PATH,
        dtype={"image_id": "string"},
        nrows=1,
    )
    test_probe = pd.read_csv(
        SAMPLE_SUBMISSION_PATH,
        dtype={"image_id": "string"},
        nrows=1,
    )
    train_filename = f"{int(train_probe.iloc[0]['image_id']):05d}.jpg"
    test_filename = f"{int(test_probe.iloc[0]['image_id']):05d}.jpg"
    TRAIN_IMAGE_DIR = find_kaggle_image_directory(
        train_filename, "train"
    )
    TEST_IMAGE_DIR = find_kaggle_image_directory(
        test_filename, "test"
    )
else:
    possible_roots = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (
            folder for folder in possible_roots
            if (folder / "data" / "train_labels.csv").exists()
        ),
        None,
    )

    if PROJECT_ROOT is None:
        raise FileNotFoundError("Project folder not found.")

    TRAIN_LABELS_PATH = PROJECT_ROOT / "data" / "train_labels.csv"
    SAMPLE_SUBMISSION_PATH = (
        PROJECT_ROOT / "data" / "sample_submission.csv"
    )
    PREPROCESSING_CONFIG_PATH = (
        PROJECT_ROOT / "outputs" / "preprocessing"
        / "preprocessing_config.json"
    )
    BASELINE_CHECKPOINT_PATH = (
        PROJECT_ROOT / "models" / "resnet18_best.pth"
    )
    TRAIN_IMAGE_DIR = PROJECT_ROOT / "data" / "train" / "train"
    TEST_IMAGE_DIR = PROJECT_ROOT / "data" / "test" / "test"

IMPROVEMENT_DIR = PROJECT_ROOT / "outputs" / "improvement"
SUBMISSION_DIR = PROJECT_ROOT / "outputs" / "submission"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
MODEL_DIR = PROJECT_ROOT / "models"
for folder in [IMPROVEMENT_DIR, SUBMISSION_DIR, FIGURE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FINAL_CHECKPOINT_PATH = MODEL_DIR / "resnet18_full_data_tta.pth"
TRAINING_HISTORY_PATH = (
    IMPROVEMENT_DIR / "full_data_training_history.csv"
)
TRAINING_SUMMARY_PATH = (
    IMPROVEMENT_DIR / "full_data_training_summary.json"
)
IMPROVEMENT_CHECKS_PATH = (
    IMPROVEMENT_DIR / "full_data_tta_checks.csv"
)
FINAL_SUBMISSION_PATH = (
    SUBMISSION_DIR / "submission_full_data_tta.csv"
)
PREDICTION_DETAILS_PATH = (
    SUBMISSION_DIR / "full_data_tta_prediction_details.csv"
)
TRAINING_FIGURE_PATH = (
    FIGURE_DIR / "figure_22_full_data_fine_tuning.png"
)
PREDICTION_FIGURE_PATH = (
    FIGURE_DIR / "figure_23_full_data_tta_predictions.png"
)
RESULTS_ZIP_PATH = PROJECT_ROOT / "resnet18_full_data_tta_results.zip"

print("Environment:", "Kaggle" if RUNNING_ON_KAGGLE else "Local")
print("Training images:", TRAIN_IMAGE_DIR)
print("Test images:", TEST_IMAGE_DIR)
print("Starting checkpoint:", BASELINE_CHECKPOINT_PATH)

## 3. Load the Complete Labelled Dataset

The model architecture and settings were already selected using the original training-validation split. The two subsets can therefore be recombined for final fitting without using any test labels.

In [ ]:
with open(PREPROCESSING_CONFIG_PATH) as config_file:
    preprocessing_config = json.load(config_file)

all_labels = pd.read_csv(
    TRAIN_LABELS_PATH,
    dtype={"image_id": "string", "class": "string"},
)
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype={"image_id": "string", "class": "string"},
)

IMAGE_SIZE = int(preprocessing_config["image_size"])
CLASS_NAMES = list(preprocessing_config["class_names"])
CLASS_TO_INDEX = dict(preprocessing_config["class_to_index"])
INDEX_TO_CLASS = {
    int(index): name for name, index in CLASS_TO_INDEX.items()
}
IMAGENET_MEAN = preprocessing_config["normalisation_mean"]
IMAGENET_STD = preprocessing_config["normalisation_std"]
AUGMENTATION = preprocessing_config["training_augmentation"]
NUMBER_OF_CLASSES = len(CLASS_NAMES)

all_labels["filename"] = all_labels["image_id"].map(
    lambda image_id: f"{int(image_id):05d}.jpg"
)
all_labels["label_index"] = all_labels["class"].map(
    CLASS_TO_INDEX
)

if all_labels["label_index"].isna().any():
    raise ValueError("An unknown class was found in train_labels.csv.")

print("Complete labelled images:", f"{len(all_labels):,}")
print("Kaggle test images:", f"{len(sample_submission):,}")
display(all_labels["class"].value_counts().reindex(CLASS_NAMES))

## 4. Configure Reproducibility and the Time Limit

Three additional epochs are deliberately conservative because the checkpoint is already highly trained. Small learning rates reduce the risk of damaging useful pretrained features. The training guard leaves time for TTA inference within a two-hour Kaggle session.

In [ ]:
RANDOM_SEED = 42
FINAL_FIT_EPOCHS = 3
BATCH_SIZE = 256
EARLY_BACKBONE_LR = 2e-6
FINAL_BLOCK_LR = 1e-5
CLASSIFIER_LR = 5e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
TRAINING_TIME_GUARD_MINUTES = 80
RUN_TRAINING = True

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2 if RUNNING_ON_KAGGLE else 0
PIN_MEMORY = DEVICE.type == "cuda"
USE_MIXED_PRECISION = DEVICE.type == "cuda"

print("Device:", DEVICE)
print("CUDA GPUs:", torch.cuda.device_count())
print("Additional epochs:", FINAL_FIT_EPOCHS)
print("Training time guard:", TRAINING_TIME_GUARD_MINUTES, "minutes")

## 5. Reuse the Established Preprocessing Pipeline

Final fitting retains the same square padding, 224 × 224 size, ImageNet normalisation, and moderate augmentation used by the validated model. Test preprocessing remains deterministic.

In [ ]:
class SquarePad:
    """Pad the shorter sides without distorting the image."""

    def __init__(self, fill=(0, 0, 0)):
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        largest = max(width, height)
        horizontal = largest - width
        vertical = largest - height
        left = horizontal // 2
        right = horizontal - left
        top = vertical // 2
        bottom = vertical - top
        return ImageOps.expand(
            image, border=(left, top, right, bottom), fill=self.fill
        )


training_transform = transforms.Compose(
    [
        SquarePad(),
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=InterpolationMode.BILINEAR,
        ),
        transforms.RandomHorizontalFlip(
            p=AUGMENTATION["horizontal_flip_probability"]
        ),
        transforms.RandomRotation(
            degrees=AUGMENTATION["maximum_rotation_degrees"],
            interpolation=InterpolationMode.BILINEAR,
            fill=(0, 0, 0),
        ),
        transforms.ColorJitter(
            brightness=AUGMENTATION["brightness_change"],
            contrast=AUGMENTATION["contrast_change"],
            saturation=AUGMENTATION["saturation_change"],
        ),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

test_transform = transforms.Compose(
    [
        SquarePad(),
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=InterpolationMode.BILINEAR,
        ),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

print("Training transform:", training_transform)
print("Test transform:", test_transform)
assert "Random" not in repr(test_transform)

## 6. Build the Full-Data and Test Loaders

The labelled loader shuffles images during fine-tuning. The test loader follows `sample_submission.csv` exactly and is never shuffled.

In [ ]:
class FacialFeatureDataset(Dataset):
    def __init__(self, dataframe, image_directory, transform, labelled):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.image_directory = Path(image_directory)
        self.transform = transform
        self.labelled = labelled

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_id = str(row["image_id"])
        filename = (
            row["filename"]
            if "filename" in self.dataframe.columns
            else f"{int(image_id):05d}.jpg"
        )
        with Image.open(self.image_directory / filename) as image:
            image = self.transform(image.convert("RGB"))

        item = {"image": image, "image_id": image_id}
        if self.labelled:
            item["label"] = int(row["label_index"])
        return item


full_training_dataset = FacialFeatureDataset(
    all_labels, TRAIN_IMAGE_DIR, training_transform, labelled=True
)
test_dataset = FacialFeatureDataset(
    sample_submission, TEST_IMAGE_DIR, test_transform, labelled=False
)

loader_options = {}
if NUM_WORKERS > 0:
    loader_options = {
        "persistent_workers": True,
        "prefetch_factor": 2,
    }

loader_generator = torch.Generator().manual_seed(RANDOM_SEED)
training_loader = DataLoader(
    full_training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=loader_generator,
    **loader_options,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    **loader_options,
)

print("Full-data batches:", len(training_loader))
print("Test batches:", len(test_loader))

## 7. Restore the Best Validated ResNet18

The experiment does not restart from ImageNet weights. It continues from the best epoch selected in Notebook 4 and verified in Notebook 5.

In [ ]:
checkpoint = torch.load(
    BASELINE_CHECKPOINT_PATH, map_location="cpu", weights_only=False
)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUMBER_OF_CLASSES)
model.load_state_dict(checkpoint["model_state_dict"])

if list(checkpoint["class_names"]) != CLASS_NAMES:
    raise ValueError("Checkpoint and configuration class order differ.")

model = model.to(DEVICE)
if DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
if DEVICE.type == "cuda" and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

print("Starting checkpoint epoch:", checkpoint["epoch"])
print("Starting validation accuracy:", f"{checkpoint['validation_accuracy']:.2%}")
print("Model GPUs:", torch.cuda.device_count() if DEVICE.type == "cuda" else 0)

## 8. Fine-Tune on All Labelled Images

Earlier layers receive the smallest learning rate, the final residual block receives a slightly larger rate, and the classifier receives the largest rate. Mixed precision reduces GPU memory use and training time.

In [ ]:
def mixed_precision_context():
    if USE_MIXED_PRECISION:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def create_optimizer(model):
    core = unwrap_model(model)
    early_parameters = chain(
        core.conv1.parameters(),
        core.bn1.parameters(),
        core.layer1.parameters(),
        core.layer2.parameters(),
        core.layer3.parameters(),
    )
    return torch.optim.AdamW(
        [
            {"params": early_parameters, "lr": EARLY_BACKBONE_LR},
            {"params": core.layer4.parameters(), "lr": FINAL_BLOCK_LR},
            {"params": core.fc.parameters(), "lr": CLASSIFIER_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def train_one_epoch(model, loader, loss_function, optimizer, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    examples = 0

    for batch in loader:
        images = batch["image"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        labels = batch["label"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        if DEVICE.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)
        with mixed_precision_context():
            logits = model(images)
            loss = loss_function(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = len(labels)
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        examples += batch_size

    return running_loss / examples, correct / examples


training_rows = []
training_start = time.perf_counter()
stop_reason = "maximum_epochs"

if RUN_TRAINING:
    optimizer = create_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=FINAL_FIT_EPOCHS, eta_min=1e-7
    )
    loss_function = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=USE_MIXED_PRECISION
    )

    for epoch in range(1, FINAL_FIT_EPOCHS + 1):
        if (time.perf_counter() - training_start) / 60 >= TRAINING_TIME_GUARD_MINUTES:
            stop_reason = "time_guard"
            print("Training stopped by the time guard.")
            break

        epoch_start = time.perf_counter()
        learning_rates = [group["lr"] for group in optimizer.param_groups]
        training_loss, training_accuracy = train_one_epoch(
            model, training_loader, loss_function, optimizer, scaler
        )
        epoch_seconds = time.perf_counter() - epoch_start

        training_rows.append(
            {
                "epoch": epoch,
                "training_loss": training_loss,
                "training_accuracy": training_accuracy,
                "early_backbone_lr": learning_rates[0],
                "final_block_lr": learning_rates[1],
                "classifier_lr": learning_rates[2],
                "epoch_minutes": epoch_seconds / 60,
                "total_minutes": (time.perf_counter() - training_start) / 60,
            }
        )
        pd.DataFrame(training_rows).to_csv(
            TRAINING_HISTORY_PATH, index=False
        )

        torch.save(
            {
                "epoch": epoch,
                "starting_checkpoint_epoch": checkpoint["epoch"],
                "model_name": "resnet18_full_data_tta",
                "model_state_dict": unwrap_model(model).state_dict(),
                "class_names": CLASS_NAMES,
                "image_size": IMAGE_SIZE,
                "labelled_images": len(full_training_dataset),
            },
            FINAL_CHECKPOINT_PATH,
        )
        scheduler.step()

        print(
            f"Epoch {epoch:02d}/{FINAL_FIT_EPOCHS} | "
            f"loss {training_loss:.4f} | "
            f"accuracy {training_accuracy:.4f} | "
            f"{epoch_seconds / 60:.1f} minutes"
        )
else:
    raise RuntimeError("RUN_TRAINING must be True for this final-fit notebook.")

training_history = pd.DataFrame(training_rows)
print("Training stop reason:", stop_reason)
display(training_history.round(5))

## 9. Record Full-Data Training Evidence

Because all labelled images are used for final fitting, this stage has no validation metric. The earlier untouched validation result remains the evidence used for model selection; this figure records only final-fit training behaviour.

In [ ]:
if training_history.empty:
    raise RuntimeError("No final-fit epoch completed.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(
    training_history["epoch"],
    training_history["training_loss"],
    marker="o",
    color="#3B82A0",
)
axes[0].set_title("Full-Data Training Loss")
axes[0].set_xlabel("Additional epoch")
axes[0].set_ylabel("Label-smoothed loss")

axes[1].plot(
    training_history["epoch"],
    training_history["training_accuracy"] * 100,
    marker="o",
    color="#009E73",
)
axes[1].set_title("Full-Data Training Accuracy")
axes[1].set_xlabel("Additional epoch")
axes[1].set_ylabel("Accuracy (%)")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.set_xticks(training_history["epoch"])

fig.suptitle(
    "ResNet18 Final Fitting on All Labelled Images",
    fontsize=15,
    weight="bold",
)
fig.tight_layout()
fig.savefig(TRAINING_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

training_summary = {
    "model": "ResNet18",
    "starting_checkpoint_epoch": int(checkpoint["epoch"]),
    "selected_validation_accuracy": float(
        checkpoint["validation_accuracy"]
    ),
    "labelled_images_used": int(len(full_training_dataset)),
    "additional_epochs_completed": int(len(training_history)),
    "final_training_loss": float(
        training_history.iloc[-1]["training_loss"]
    ),
    "final_training_accuracy": float(
        training_history.iloc[-1]["training_accuracy"]
    ),
    "total_training_minutes": float(
        training_history.iloc[-1]["total_minutes"]
    ),
    "stop_reason": stop_reason,
    "test_images_used_for_training": False,
    "validation_reused_after_model_selection": True,
}
with open(TRAINING_SUMMARY_PATH, "w") as summary_file:
    json.dump(training_summary, summary_file, indent=4)

print("Training evidence:", TRAINING_HISTORY_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)

## 10. Generate Two-View Test-Time-Augmented Predictions

For each test image, the model predicts the original tensor and a horizontal mirror of the same tensor. Their logits are averaged before choosing the class. This does not train the model; it reduces sensitivity to left-right orientation.

In [ ]:
model.eval()
prediction_rows = []
prediction_start = time.perf_counter()

with torch.inference_mode():
    for batch in test_loader:
        images = batch["image"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        if DEVICE.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)

        with mixed_precision_context():
            original_logits = model(images)
            flipped_logits = model(torch.flip(images, dims=[3]))
            averaged_logits = (original_logits + flipped_logits) / 2
            probabilities = torch.softmax(averaged_logits, dim=1)

        confidence, predicted_indices = probabilities.max(dim=1)
        for image_id, class_index, probability in zip(
            batch["image_id"],
            predicted_indices.cpu().tolist(),
            confidence.cpu().tolist(),
        ):
            prediction_rows.append(
                {
                    "image_id": str(image_id),
                    "predicted_class": INDEX_TO_CLASS[class_index],
                    "confidence": probability,
                }
            )

prediction_seconds = time.perf_counter() - prediction_start
prediction_details = pd.DataFrame(prediction_rows)
print("TTA predictions:", f"{len(prediction_details):,}")
print("TTA prediction time:", f"{prediction_seconds / 60:.2f} minutes")
display(prediction_details.head(10))

## 11. Build and Validate the Improved Submission

The submission preserves the exact template order and contains only the required columns. Every row and class name is checked before the file is saved.

In [ ]:
submission = sample_submission[["image_id"]].copy()
submission["class"] = prediction_details["predicted_class"].values
submission.to_csv(FINAL_SUBMISSION_PATH, index=False)
prediction_details.to_csv(PREDICTION_DETAILS_PATH, index=False)

checks = pd.DataFrame(
    {
        "check": [
            "All 63,660 labelled images used",
            "At least one final-fit epoch completed",
            "Final checkpoint exists",
            "Submission columns are exact",
            "Submission row count is exact",
            "Image ID order is unchanged",
            "Image IDs are unique",
            "No prediction is missing",
            "All labels are official classes",
            "Test images were not used for training",
            "Two-view TTA was used",
        ],
        "result": [
            len(full_training_dataset) == 63660,
            len(training_history) >= 1,
            FINAL_CHECKPOINT_PATH.exists(),
            list(submission.columns) == ["image_id", "class"],
            len(submission) == len(sample_submission),
            submission["image_id"].equals(sample_submission["image_id"]),
            submission["image_id"].is_unique,
            not submission["class"].isna().any(),
            set(submission["class"]).issubset(set(CLASS_NAMES)),
            training_summary["test_images_used_for_training"] is False,
            True,
        ],
        "expected": [True] * 11,
    }
)
checks.to_csv(IMPROVEMENT_CHECKS_PATH, index=False)
display(checks)
assert checks["result"].all()
print("Improved submission:", FINAL_SUBMISSION_PATH)

## 12. Save Prediction Evidence

The prediction distribution and confidence histogram document the improved pipeline's output. These figures are descriptive and are not used to tune the model.

In [ ]:
predicted_counts = (
    submission["class"].value_counts().reindex(CLASS_NAMES, fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
bars = axes[0].bar(
    predicted_counts.index,
    predicted_counts.values,
    color="#3B82A0",
    edgecolor="#243447",
)
axes[0].bar_label(bars, padding=3, fontsize=8)
axes[0].set_title("Full-Data + TTA Predictions by Class")
axes[0].set_ylabel("Test images")
axes[0].tick_params(axis="x", rotation=30)

axes[1].hist(
    prediction_details["confidence"],
    bins=20,
    color="#E69F00",
    edgecolor="#243447",
)
axes[1].set_title("Full-Data + TTA Prediction Confidence")
axes[1].set_xlabel("Highest predicted probability")
axes[1].set_ylabel("Test images")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)

fig.suptitle("Improved ResNet18 Test Prediction Evidence", weight="bold")
fig.tight_layout()
fig.savefig(PREDICTION_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

prediction_summary = {
    "method": "ResNet18 full labelled data plus horizontal-flip TTA",
    "baseline_kaggle_accuracy": 0.9463,
    "test_images": int(len(submission)),
    "prediction_seconds": float(prediction_seconds),
    "mean_confidence": float(prediction_details["confidence"].mean()),
    "minimum_confidence": float(prediction_details["confidence"].min()),
    "predicted_class_counts": {
        name: int(value) for name, value in predicted_counts.items()
    },
    "submission_checks_passed": bool(checks["result"].all()),
}
PREDICTION_SUMMARY_PATH = (
    IMPROVEMENT_DIR / "full_data_tta_prediction_summary.json"
)
with open(PREDICTION_SUMMARY_PATH, "w") as summary_file:
    json.dump(prediction_summary, summary_file, indent=4)

display(pd.DataFrame({
    "class": CLASS_NAMES,
    "predictions": predicted_counts.values,
    "percentage": predicted_counts.values / len(submission) * 100,
}).round(2))

## 13. Package and Download the Improved Results

Download the improved submission, results ZIP, and executed notebook. Submit only `submission_full_data_tta.csv` to Kaggle and record its score before deciding whether to change the final-leaderboard selection.

In [ ]:
files_to_package = [
    FINAL_CHECKPOINT_PATH,
    TRAINING_HISTORY_PATH,
    TRAINING_SUMMARY_PATH,
    IMPROVEMENT_CHECKS_PATH,
    FINAL_SUBMISSION_PATH,
    PREDICTION_DETAILS_PATH,
    PREDICTION_SUMMARY_PATH,
    TRAINING_FIGURE_PATH,
    PREDICTION_FIGURE_PATH,
]

with ZipFile(RESULTS_ZIP_PATH, "w", ZIP_DEFLATED) as archive:
    for path in files_to_package:
        archive.write(path, arcname=path.relative_to(PROJECT_ROOT))

print("Results package:", RESULTS_ZIP_PATH)
print("Improved Kaggle CSV:", FINAL_SUBMISSION_PATH)

if RUNNING_ON_KAGGLE:
    csv_link = PROJECT_ROOT / FINAL_SUBMISSION_PATH.name
    shutil.copy2(FINAL_SUBMISSION_PATH, csv_link)
    display(FileLink(csv_link.name))
    display(FileLink(RESULTS_ZIP_PATH.name))

## 14. Conclusion

The selected ResNet18 was fine-tuned on all available labelled images after validation-based model selection was complete. The final prediction pipeline then averaged the original and horizontally flipped views of each test image. The resulting CSV passed all structural and class-label checks.

The new Kaggle accuracy must be compared with the original 0.9463 score. A higher score would provide direct evidence that full-data final fitting and test-time augmentation improved hidden-test generalisation.